In [ ]:
import torch
from torch_geometric.data import HeteroData

PROCESSED_DIR = '../data/processed/'

In [ ]:
# 1. Load Pre-computed Artifacts

print("Loading node features...")
x_user = torch.load(PROCESSED_DIR + 'x_dict_user.pt', weights_only=True)
x_item = torch.load(PROCESSED_DIR + 'x_dict_item.pt', weights_only=True)

print("Loading edge indices...")
user_item_edge_index = torch.load(PROCESSED_DIR + 'user_item_edge_index.pt', weights_only=True)
item_item_edge_index = torch.load(PROCESSED_DIR + 'item_item_edge_index.pt', weights_only=True)

print(f"  x_user shape          : {x_user.shape}")
print(f"  x_item shape          : {x_item.shape}")
print(f"  user->item edge_index : {user_item_edge_index.shape}")
print(f"  item->item edge_index : {item_item_edge_index.shape}")

In [ ]:
# 2. Construct `HeteroData` Object

data = HeteroData()

# --- Node features ---
data['user'].x = x_user   
data['item'].x = x_item   

# --- Edge indices ---
# user --reviews--> item
data['user', 'reviews', 'item'].edge_index = user_item_edge_index   

# item --also_bought--> item
data['item', 'also_bought', 'item'].edge_index = item_item_edge_index  

print(data)

In [ ]:
# 3. Validate the Graph

print("=== HeteroData Summary ===")
print(f"Node types  : {data.node_types}")
print(f"Edge types  : {data.edge_types}")
print()
print(f"data['user'].x          : {data['user'].x.shape}")
print(f"data['item'].x          : {data['item'].x.shape}")
print()
print(f"data['user','reviews','item'].edge_index    : {data['user', 'reviews', 'item'].edge_index.shape}")
print(f"data['item','also_bought','item'].edge_index: {data['item', 'also_bought', 'item'].edge_index.shape}")
print()

# Sanity checks
num_users = data['user'].x.shape[0]
num_items = data['item'].x.shape[0]

ui_idx = data['user', 'reviews', 'item'].edge_index
ii_idx = data['item', 'also_bought', 'item'].edge_index

assert ui_idx.shape[0] == 2, "user->item edge_index must have shape [2, E]"
assert ii_idx.shape[0] == 2, "item->item edge_index must have shape [2, E]"
assert ui_idx[0].max() < num_users, "user node index out of range"
assert ui_idx[1].max() < num_items, "item node index out of range (user->item)"
assert ii_idx[0].max() < num_items, "item src index out of range (item->item)"
assert ii_idx[1].max() < num_items, "item dst index out of range (item->item)"

print("All sanity checks passed!")

In [ ]:
# 4. Save the HeteroData Object

save_path = PROCESSED_DIR + 'hetero_data.pt'
torch.save(data, save_path)
print(f"HeteroData object saved to {save_path}")

In [ ]:
# 5. Schema Visualization

import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Build a directed schema graph from node/edge types
G = nx.DiGraph()
for node_type in data.node_types:
    G.add_node(node_type)
for src, rel, dst in data.edge_types:
    G.add_edge(src, dst, label=rel)

pos = nx.spring_layout(G, seed=42)
edge_labels = {(u, v): d["label"] for u, v, d in G.edges(data=True)}

node_colors = {"user": "#4A90D9", "item": "#E87040"}
colors = [node_colors.get(n, "#888888") for n in G.nodes]

fig, ax = plt.subplots(figsize=(7, 4))
nx.draw(
    G, pos, ax=ax,
    with_labels=True,
    node_color=colors,
    node_size=4000,
    font_color="white",
    font_size=14,
    font_weight="bold",
    arrows=True,
    arrowsize=30,
    connectionstyle="arc3,rad=0.15",
)
nx.draw_networkx_edge_labels(
    G, pos, edge_labels=edge_labels,
    ax=ax, font_size=11, font_color="#333333",
    bbox=dict(boxstyle="round,pad=0.3", fc="#f5f5f5", ec="#cccccc", alpha=0.9),
)

legend = [mpatches.Patch(color=c, label=n) for n, c in node_colors.items()]
ax.legend(handles=legend, loc="upper right", fontsize=11)
ax.set_title("HeteroData Graph Schema", fontsize=15, fontweight="bold", pad=14)
plt.tight_layout()
plt.show()
